# How to read a Parquet file with Python

Load Parquet into pandas, read only the columns and rows you need, look inside a file before reading it, stream files larger than memory, and query Parquet with SQL.

This notebook accompanies [How to read a Parquet file with Python](https://docs.opteryx.app/docs/guides/read-a-parquet-file-with-python) on docs.opteryx.app. It builds its own sample data, so it runs top to bottom with nothing to download.

In [ ]:
%pip install -q pandas pyarrow rugo opteryx-core

## Make some sample data

200,000 synthetic orders, sorted by `ordered_at`. Every example below reads this file.

In [ ]:
import datetime
import random

import pyarrow

random.seed(42)
N = 200_000
start = datetime.datetime(2024, 1, 1)

orders = pyarrow.table({
    "order_id": list(range(1, N + 1)),
    "customer_id": [random.randint(1, 20_000) for _ in range(N)],
    "country": [random.choice(["GB", "US", "DE", "FR", "JP", "AU", "CA", "NL"]) for _ in range(N)],
    "status": random.choices(["delivered", "shipped", "pending", "cancelled", "returned"], weights=[60, 15, 10, 10, 5], k=N),
    "amount": [round(random.lognormvariate(3.5, 0.8), 2) for _ in range(N)],
    "ordered_at": sorted(start + datetime.timedelta(seconds=random.randint(0, 365 * 86400)) for _ in range(N)),
})
orders.num_rows

In [ ]:
import pyarrow.parquet as pq

# ten row groups of 20,000 rows each
pq.write_table(orders, "orders.parquet", row_group_size=20_000)

## The short answer: pandas

pandas hands the read to PyArrow. Types come from the schema stored in the file.

In [ ]:
import pandas as pd

df = pd.read_parquet("orders.parquet")
df.dtypes

In [ ]:
df.head()

## Read only the columns you need

A column you don't ask for is never read from disk or decompressed.

In [ ]:
df = pd.read_parquet("orders.parquet", columns=["order_id", "country", "amount"])
df.shape

## Filter while reading

A list of tuples is AND.

In [ ]:
df = pd.read_parquet(
    "orders.parquet",
    columns=["order_id", "country", "amount"],
    filters=[("country", "==", "GB"), ("amount", ">", 100)],
)
len(df)

A list of lists is OR, and `in` is usually simpler.

In [ ]:
df = pd.read_parquet(
    "orders.parquet",
    filters=[[("status", "==", "returned")], [("status", "==", "cancelled")]],
)
len(df)

In [ ]:
df = pd.read_parquet("orders.parquet", filters=[("status", "in", ["returned", "cancelled"])])
len(df)

Compare timestamps against a `pd.Timestamp`, not a string.

In [ ]:
df = pd.read_parquet("orders.parquet", filters=[("ordered_at", ">=", pd.Timestamp("2024-12-01"))])
len(df)

## Why filtering on read is fast

Each row group stores min/max statistics. Because the file is sorted by `ordered_at`, each row group covers about five weeks, so the December filter above only needs the last one. `country` isn't sorted, so every row group contains every country and nothing can be skipped for a country filter.

In [ ]:
pf = pq.ParquetFile("orders.parquet")

for i in range(pf.metadata.num_row_groups):
    stats = pf.metadata.row_group(i).column(5).statistics   # ordered_at
    print(i, stats.min, "->", stats.max)

## Look inside before you read

The footer holds the schema, row counts and statistics. Reading it doesn't touch the data.

In [ ]:
print(pf.schema_arrow)
print(pf.metadata.num_rows, "rows in", pf.metadata.num_row_groups, "row groups")

## Files bigger than memory

`iter_batches` keeps one batch in memory at a time.

In [ ]:
totals = {}
for batch in pf.iter_batches(batch_size=50_000, columns=["country", "amount"]):
    chunk = batch.to_pandas()
    for country, amount in chunk.groupby("country")["amount"].sum().items():
        totals[country] = totals.get(country, 0) + amount

pd.Series(totals).sort_values(ascending=False).round(2)

## Reading a folder of files

Pass a folder instead of a file. Hive-style partition folders become a column, and filters on it skip whole folders.

In [ ]:
import pyarrow

orders_with_month = orders.append_column(
    "month", pyarrow.array(orders["ordered_at"].to_pandas().dt.strftime("%Y-%m"))
)
pq.write_to_dataset(orders_with_month, "orders_by_month", partition_cols=["month"])

df = pd.read_parquet("orders_by_month", filters=[("month", "==", "2024-12")])
len(df)

## Without PyArrow: Rugo

[Rugo](https://docs.opteryx.app/docs/guides/rugo-standalone) is the reader inside Opteryx. It prunes row groups and filters rows as it reads. (The conversion to pandas goes through Arrow.)

In [ ]:
from rugo import parquet

with parquet.read_parquet(
    "orders.parquet",
    columns=["order_id", "country", "amount"],
    predicates=[("country", "=", "GB"), ("amount", ">", 100)],
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

len(df)

## With SQL: Opteryx

Aggregate inside the engine and only bring the answer into pandas.

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT country,
           COUNT(*)              AS orders,
           ROUND(AVG(amount), 2) AS avg_amount
      FROM READ_PARQUET('orders.parquet')
     WHERE status = 'delivered'
       AND ordered_at >= CAST('2024-07-01' AS TIMESTAMP)
     GROUP BY country
     ORDER BY orders DESC
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

In [ ]:
# bind values as parameters rather than formatting them into the SQL
morsels = opteryx.session().execute_to_morsels(
    "SELECT order_id, amount FROM READ_PARQUET('orders.parquet') WHERE country = :country AND amount > :min_amount",
    params={"country": "GB", "min_amount": 100},
)
len(pyarrow.concat_tables(m.to_arrow() for m in morsels))